# Chapter 3 · Lab 1 — Diagnose the performance gap

Read [background.md](../background.md), then run cells top to bottom in a fresh kernel.
Prerequisites: Chapter 1's validated Qwen3/cache and checkpoint loader; Chapter 2's
FLOP/byte ledger, roofline, CUDA event timing and frozen predictions. This lab adds
reusable [profiling.py](profiling.py) and [study.py](study.py), used again by Labs 2/3.
It requires the pinned local **8B and 32B checkpoints**, BF16-capable GB10, and the
[course environment](../../../shared/SETUP.md). Use `uv sync --extra kernels`.
CPU oracle results are not model GPU measurements.

Follow **prediction → implementation → correctness → measurement → explanation**.
The complete driver is runnable; exercises change its capture scope and interpret
actual evidence. Loading, validation and compilation stay outside captured calls.

In [ ]:
from pathlib import Path
import importlib, json, os, sys, time
from uuid import uuid4
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'pyproject.toml').is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
# Keep compiler disk caches from different DSL versions separate. In-memory JIT reuse remains enabled.
os.environ.setdefault('CUTE_DSL_DISABLE_FILE_CACHING', '1')
from IPython.display import display, Markdown, Image
from shared import performance as perf
study = importlib.import_module('chapters.03_kernels.code.study')
plots = importlib.import_module('chapters.03_kernels.code.plots')
RUN_GPU = True
MODEL_KEYS = ['8b', '32b']
EXTERNAL_PROFILERS = True
# Set P03_REPLAY_DIR only to inspect an existing run. Replay never creates measurements.
REPLAY_DIR = os.environ.get('P03_REPLAY_DIR')

## 1. Freeze the prediction and identify the forward boundary

For each model, use batches 1 and 4, prompt length 2048, empty cache before prefill,
and prefix length 2048 before the first decode append. Derive the shapes of QK,
expanded KV and SwiGLU intermediates. Reproduce 32B's rectangular Q projection:
`[B*T,5120] @ [5120,8192]`. Verify dimensions against local config files.

The next cell shows Chapter 2 predictions. `study.execute` writes all per-step
predictions and source hashes **before** loading models or taking measurements.
The assumed dense BF16 ceiling is 125 TFLOP/s; 273 GB/s is the labeled hardware
bandwidth. These ideal bounds exclude launch, allocation and extra attention traffic.
Local TTFT includes prefill plus greedy token selection; events bracket only forward.

In [ ]:
for key in MODEL_KEYS:
    print(key, perf.MODELS[key])
    display(perf.predict_tradeoffs(perf.MODELS[key], [1,4], 2048, study.hardware(), 'bfloat16'))
# Exercise: write predicted bottleneck, bytes saved, and latency reduction before the run.
hypotheses = [dict(operation=name, evidence_to_seek=evidence, predicted_local_speedup=2.0,
                   rationale='Replace this illustrative 2x assumption with your derivation')
              for name,evidence in [('attention','score/softmax allocations and repeat_interleave'),
                                    ('normalize','square, mean, rsqrt, scale launches'),
                                    ('activation','SiLU temporary and multiply'),
                                    ('cache','cat allocation/copy'),('host','gaps between short launches')]]
display(hypotheses)

## 2. Learn the tools in order

1. **[PyTorch Profiler](https://docs.pytorch.org/docs/stable/profiler.html)**:
   correlate Python operators, shapes, allocated tensors and CUDA kernels. Inspect
   `operators.csv`, `operators.txt`, `trace.json`, and `kernel_summary.json` for every
   model/batch/phase. Open traces in a compatible trace viewer. Locate
   `repeat_interleave`, matmul, softmax, norm/activation, and concatenation.
2. **[Nsight Systems](https://docs.nvidia.com/nsight-systems/UserGuide/)**:
   follow `prefill_B1_S2048`, `decode_B1_S2048`, and batch-4 NVTX ranges. Read
   kernel launches, host dispatch, synchronization, copies and idle gaps. Open
   `report.nsys-rep`; inspect `summary.csv` (`cuda_gpu_kern_sum`, `cuda_api_sum`,
   `nvtx_sum`). Separate host overhead from kernel execution.
3. **[Nsight Compute](https://docs.nvidia.com/nsight-compute/ProfilingGuide/)**:
   select the attention/RMS/softmax kernels identified above. Inspect DRAM bytes,
   occupancy, registers and compute activity. `command.log` saves exact commands;
   `status.json` preserves counter permission failures. On the preparation GB10,
   `ERR_NVGPUCTRPERM` prevents this evidence. Continue other measurements and mark
   counters incomplete; administrator access is an external prerequisite.

Each tool runs separately. `cudaProfilerStart/Stop` scopes external capture after
all cases warm; NVTX identifies each case. No profiler timings enter `results.csv`.

In [ ]:
# Driver interface: model / implementation / batch / context / phase / profiler.
import subprocess
print(subprocess.check_output([sys.executable, str(ROOT/'chapters/03_kernels/code/study.py'), '--help'], text=True))

## 3. Establish unprofiled baselines, validate, then capture

The driver loads models sequentially, checks full-prefix versus cached/chunked
logits and cache ownership, then runs two complete warmups and three repeats,
each with eight decode calls. Prompt IDs, forced continuation, revisions and
precision are saved. Greedy selection remains inside the wall timing boundary,
but both implementations will receive identical forced history for controlled comparison.

**Implementation exercise:** inspect `prepare_call`. Explain why repeatedly appending
into a shared mutable cache would invalidate its fixed-prefix decode capture.
Add a new `context=129` capture to exercise a partial tile, preserving the original
2048 cases. Predict which intermediates scale quadratically and which linearly.

In [ ]:
if RUN_GPU:
    if REPLAY_DIR:
        RUN_DIR = Path(REPLAY_DIR).resolve()
        print('Replaying saved evidence; no new measurements:', RUN_DIR)
        assert (RUN_DIR/'completion.json').is_file()
    else:
        RUN_DIR = ROOT/'results'/('p03-baseline-' + time.strftime('%Y%m%d-%H%M%S') + '-' + uuid4().hex[:6])
        study.execute(RUN_DIR, 'baseline', MODEL_KEYS, external=EXTERNAL_PROFILERS)
    raw = plots.read_rows(RUN_DIR/'results.csv')
    summary = perf.summarize_model_rows(raw)
    display(Markdown(f'Artifacts: `{RUN_DIR}` · {len(raw)} unprofiled observations'))
    display(json.loads((RUN_DIR/'status.json').read_text()))
else:
    print('GPU/checkpoint work unmeasured. Enable RUN_GPU after setup.')

## 4. Rank mechanisms using evidence

For each case record a trace/operator range, proposed mechanism, expected
traffic/launch reduction, baseline fraction and predicted integrated benefit.
`f` must refer to one exclusive scope; do not add inclusive parent and child time.
Use `1 / ((1-f) + f/s + adapter_ms/baseline_ms)`. Profile-derived fractions are
perturbed estimates, so compare the prediction against independent ordinary timing.
Dynamic cache concatenation and host overhead remain for later chapters.

In [ ]:
ranked_optimizations = [dict(rank=i+1, operation=h['operation'], evidence=None,
    traffic_or_launch_reduction=None, baseline_time_fraction=None,
    predicted_local_speedup=h['predicted_local_speedup'], predicted_integrated_speedup=None)
    for i,h in enumerate(hypotheses)]
display(ranked_optimizations)
# Fill from actual traces before Lab 2. None means not yet supported, never zero cost.
if RUN_GPU:
    perf.write_json(RUN_DIR/'student_ranked_optimizations.json', ranked_optimizations)
    for key in MODEL_KEYS:
        for batch in [1,4]:
            for phase in ['prefill','decode']:
                path = RUN_DIR/f'{key}-baseline-B{batch}-S2048-{phase}'/'torch/kernel_summary.json'
                if path.exists():
                    print(key, batch, phase, json.loads(path.read_text())['count'], 'CUDA kernels')
if RUN_GPU:
    display(json.loads((RUN_DIR/'ranked_optimizations.json').read_text()))


## Completion and explanation

Complete both models × both batches × both phases. Retain frozen predictions,
raw unprofiled repeats, correctness, all PyTorch traces/operator tables, Nsight
Systems reports/summaries, and targeted Nsight Compute reports or failure logs.
Explain five candidates: materialized scores, repeated KV heads, normalization/
activation launches, cache concatenation, and host overhead. Rank them using
measured evidence, not intuition. Reproduce one FLOP and one traffic calculation
without code and defend which cost the proposed fusion cannot remove.
Continue to [Lab 2](lab2.ipynb). Counter access can remain explicitly incomplete.